In [1]:
#clear all
%reset -f

#import packages
import numpy as np
import scipy
import sys
import os
import pandas as pd
import mne
import matplotlib
from sklearn.utils import resample
from mne_icalabel import label_components

root = 'F:/Documents/Science/MirRevAdaptEEG'
participants = list(range(0,32))
#specify which erp we are analyzing
erps = 'frn'

#pop up plots as separate window & interactive
%matplotlib qt
matplotlib.pyplot.close('all')

In [2]:
# First, we get the 800 timepoints we are considering as a list, to use for indices given by cluster-based permutation later
# But only grab 0 to 1 sec time-locked to feedback onset (idx= 400:601)
root_directory = root
data_directory = os.path.join(root_directory, 'data/eeg/')
pp = 0 #only need one participant

# we can use aligned data

id_directory = os.path.join(data_directory, 'p%03d/' % pp)
pp_directory = os.path.join(id_directory, erps)
fname = os.path.join(pp_directory, 'p%03d_%s_%s-ave.fif' % (pp, 'early_late', 'aligned'))
evoked = mne.read_evokeds(fname)
df = evoked[0].to_data_frame()
time = df['time'].tolist()
time = time[400:601] #get only timepoints we want

Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p000\frn\p000_early_late_aligned-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 48 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)


In [3]:
# Load data for each condition
# Take the mean across electrodes of interest per participant
# transform data into array accepted for cluster-based permutation test in mne
# output: (n_participants, n_timepts) for each condition

root_directory = root
data_directory = os.path.join(root_directory, 'data/eeg/')

#specify channels we need - FRN
channels = ['FCz', 'F3', 'Fz', 'F4', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4']

    
#read in evoked object
conditionnames = ['early1', 'late2', 'early2', 'late1']


#aligned
evoked_list = []

for pp in participants:
    id_directory = os.path.join(data_directory, 'p%03d/' % participants[pp])
    pp_directory = os.path.join(id_directory, erps)
    fname = os.path.join(pp_directory, 'p%03d_%s_%s-ave.fif' % (participants[pp], 'early_late', 'aligned'))
    evoked = mne.read_evokeds(fname)
    evoked = evoked[0]
    evoked = evoked.get_data(picks=channels) #will give data of shape (n_channels, n_timepts)
    evoked = evoked.mean(axis=0) #take mean of cols or the channels we picked
    evoked = evoked[400:601] #get timepoints we need
    evoked_list.append(evoked)
    
aligned_flist = evoked_list

#rdm

for condition in range(0, len(conditionnames)):
    evoked_list = []
    for pp in participants:
        id_directory = os.path.join(data_directory, 'p%03d/' % participants[pp])
        pp_directory = os.path.join(id_directory, erps)
        fname = os.path.join(pp_directory, 'p%03d_%s_%s-ave.fif' % (participants[pp], conditionnames[condition], 'rdm'))
        evoked = mne.read_evokeds(fname)
        evoked = evoked[0]
        evoked = evoked.get_data(picks=channels) #will give data of shape (n_channels, n_timepts)
        evoked = evoked.mean(axis=0) #take mean of cols or the channels we picked
        evoked = evoked[400:601] #get timepoints we need
        evoked_list.append(evoked)
    if condition == 0:
        early1_rdm_flist = evoked_list
    elif condition == 1:
        late2_rdm_flist = evoked_list
    elif condition == 2:
        early2_rdm_flist = evoked_list
    elif condition == 3:
        late1_rdm_flist = evoked_list

Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p000\frn\p000_early_late_aligned-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 48 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p001\frn\p001_early_late_aligned-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 48 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p002\frn\p002_early_late_aligned-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16

Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p020\frn\p020_early_late_aligned-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 48 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p021\frn\p021_early_late_aligned-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 48 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p022\frn\p022_early_late_aligned-ave.fif ...
    Found

Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p008\frn\p008_early1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p009\frn\p009_early1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p010\frn\p010_early1_rdm-ave.fif ...
    Found the data of interest:
 

    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p029\frn\p029_early1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p030\frn\p030_early1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified fo

        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p017\frn\p017_late2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p018\frn\p018_late2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Read

Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p005\frn\p005_early2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p006\frn\p006_early2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p007\frn\p007_early2_rdm-ave.fif ...
    Found the data of interest:
 

    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p026\frn\p026_early2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p027\frn\p027_early2_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 12 - aspect type = 100
No projector specified fo

        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p014\frn\p014_late1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Reading F:\Documents\Science\MirRevAdaptEEG\data\eeg\p015\frn\p015_late1_rdm-ave.fif ...
    Found the data of interest:
        t =   -2000.00 ...    1995.00 ms (16142)
        0 CTF compensation matrices available
        nave = 24 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
Loaded Evoked data is baseline-corrected (baseline: [-0.1, 0] sec)
Read

In [4]:
# Comparing two ERP signals is just the same as taking their difference (ERP1 minus ERP2) and using this in permutation test
def get_clust_perm_test(conditionA, conditionB, pval, n_permutations):
    #take difference of two conditions
    data = np.subtract(conditionA, conditionB)
    np.shape(data)
    #define cluster forming threshold based on p-value
    df = len(participants) - 1
    thresh = scipy.stats.t.ppf(1 - pval / 2, df)
    #run cluster-based permutation test
    T_0, clust_idx, clust_pvals, H0 = mne.stats.permutation_cluster_1samp_test(data, threshold = thresh, 
                                                          n_permutations = n_permutations, tail = 0, 
                                                          adjacency = None, seed = 999, 
                                                          out_type = 'mask', verbose = True)

    return T_0, clust_idx, clust_pvals, H0

In [5]:
# First, we can compare each condition to aligned
# Generate a data frame to tabulate condition, cluster indices, cluster timepts, p values
# This information can then be included in ERP plots

flists = [early1_rdm_flist, late2_rdm_flist, early2_rdm_flist, late1_rdm_flist]
conditionnames = ['early1rdm', 'late2rdm', 'early2rdm', 'late1rdm']
p = 0.05
perms = 1000

condition = []
clust_idx_start = []
clust_idx_end = []
time_start = []
time_end = []
p_values = []

for f in range(0, len(flists)):
    T_0, clust_idx, clust_pvals, H0 = get_clust_perm_test(flists[f], aligned_flist, p, perms)
#     print(clust_idx, clust_pvals)
    if len(clust_idx) == 0:
        condition.append(conditionnames[f])
        clust_idx_start.append(np.nan)
        clust_idx_end.append(np.nan)
        time_start.append(np.nan)
        time_end.append(np.nan)
        p_values.append(np.nan)
    else:
        for clust in range(0, len(clust_idx)):
            cluster = clust_idx[clust][0] #to get the slice sequence we need
        
            cluster_start = cluster.start
            clust_idx_start.append(cluster_start)
        
            cluster_end = cluster.stop
            clust_idx_end.append(cluster_end)
        
            time_idx_start = time[cluster_start]
            time_start.append(time_idx_start)
        
            time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
            time_end.append(time_idx_end)
        
            clust_p = clust_pvals[clust]
            p_values.append(clust_p)
        
            condition.append(conditionnames[f])
        
perm_test = pd.DataFrame(
    {'condition': condition,
     'clust_idx_start': clust_idx_start,
     'clust_idx_end': clust_idx_end,
     'time_start': time_start,
     'time_end': time_end,
     'p_values': p_values})

perm_test_filename = os.path.join('F:/Documents/Science/MirRevAdaptEEG/data/', 'Permutation_test_RDMvsAligned_%s.csv' % (erps))
perm_test.to_csv(perm_test_filename)

stat_fun(H1): min=0.855675 max=7.090056
Running initial clustering …
Found 6 clusters


  0%|          | Permuting : 0/999 [00:00<?,       ?it/s]

stat_fun(H1): min=-0.062957 max=7.592397
Running initial clustering …
Found 3 clusters


  0%|          | Permuting : 0/999 [00:00<?,       ?it/s]

stat_fun(H1): min=1.266351 max=7.478404
Running initial clustering …
Found 3 clusters


  0%|          | Permuting : 0/999 [00:00<?,       ?it/s]

stat_fun(H1): min=-0.010348 max=7.426523
Running initial clustering …
Found 2 clusters


  0%|          | Permuting : 0/999 [00:00<?,       ?it/s]

In [7]:
# Next, we subtract aligned from each condition, so that we can compare early vs late in each perturbation type
diffconds = ['early1rdm', 'late2rdm', 'early2rdm', 'late1rdm']
early1rdm_diff = []
late2rdm_diff = []
early2rdm_diff = []
late1rdm_diff = []

for cond in range(0, len(diffconds)):
    if cond == 0:
        diffevks = np.subtract(early1_rdm_flist, aligned_flist)
        early1rdm_diff.append(diffevks)
        early1rdm_diff = early1rdm_diff[0] #to keep shape of object consistent
    elif cond == 1:
        diffevks = np.subtract(late2_rdm_flist, aligned_flist)
        late2rdm_diff.append(diffevks)
        late2rdm_diff = late2rdm_diff[0]
    elif cond == 2:
        diffevks = np.subtract(early2_rdm_flist, aligned_flist)
        early2rdm_diff.append(diffevks)
        early2rdm_diff = early2rdm_diff[0]
    elif cond == 3:
        diffevks = np.subtract(late1_rdm_flist, aligned_flist)
        late1rdm_diff.append(diffevks)
        late1rdm_diff = late1rdm_diff[0]

In [8]:
# Generate a data frame to tabulate condition, cluster indices, cluster timepts, p values of EARLY VS LATE comparisons
# This information can then be included in ERP difference wave plots

conds = ['rdm1_diff', 'rdm2_diff']
conditionnames = ['rdm1', 'rdm2']
p = 0.05
perms = 1000

condition = []
clust_idx_start = []
clust_idx_end = []
time_start = []
time_end = []
p_values = []

for c in range(0, len(conds)):
    if c == 0:
        T_0, clust_idx, clust_pvals, H0 = get_clust_perm_test(late1rdm_diff, early1rdm_diff, p, perms)
#         print(clust_idx, clust_pvals)
        if len(clust_idx) == 0:
            condition.append(conditionnames[c])
            clust_idx_start.append(np.nan)
            clust_idx_end.append(np.nan)
            time_start.append(np.nan)
            time_end.append(np.nan)
            p_values.append(np.nan)
        else:
            for clust in range(0, len(clust_idx)):
                cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                cluster_start = cluster.start
                clust_idx_start.append(cluster_start)
        
                cluster_end = cluster.stop
                clust_idx_end.append(cluster_end)
        
                time_idx_start = time[cluster_start]
                time_start.append(time_idx_start)
        
                time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                time_end.append(time_idx_end)
        
                clust_p = clust_pvals[clust]
                p_values.append(clust_p)
        
                condition.append(conditionnames[c])
            
    elif c == 1:
        T_0, clust_idx, clust_pvals, H0 = get_clust_perm_test(late2rdm_diff, early2rdm_diff, p, perms)
        if len(clust_idx) == 0:
            condition.append(conditionnames[c])
            clust_idx_start.append(np.nan)
            clust_idx_end.append(np.nan)
            time_start.append(np.nan)
            time_end.append(np.nan)
            p_values.append(np.nan)
        else:    
            for clust in range(0, len(clust_idx)):
                cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                cluster_start = cluster.start
                clust_idx_start.append(cluster_start)
        
                cluster_end = cluster.stop
                clust_idx_end.append(cluster_end)
        
                time_idx_start = time[cluster_start]
                time_start.append(time_idx_start)
        
                time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                time_end.append(time_idx_end)
        
                clust_p = clust_pvals[clust]
                p_values.append(clust_p)
        
                condition.append(conditionnames[c])
        
perm_test = pd.DataFrame(
    {'condition': condition,
     'clust_idx_start': clust_idx_start,
     'clust_idx_end': clust_idx_end,
     'time_start': time_start,
     'time_end': time_end,
     'p_values': p_values})

perm_test_filename = os.path.join('F:/Documents/Science/MirRevAdaptEEG/data/', 'Permutation_test_RDMEarlyvsLate_%s.csv' % (erps))
perm_test.to_csv(perm_test_filename)

stat_fun(H1): min=-4.438228 max=0.940138
Running initial clustering …
Found 2 clusters


  0%|          | Permuting : 0/999 [00:00<?,       ?it/s]

stat_fun(H1): min=-3.056901 max=-0.563957
Running initial clustering …
Found 4 clusters


  0%|          | Permuting : 0/999 [00:00<?,       ?it/s]

In [9]:
# Next step is to subtract early from late condition, to generate a single signal for each perturbation
diffconds = ['rdm1', 'rdm2']
rdm1_diff = []
rdm2_diff = []

for cond in range(0, len(diffconds)):
    if cond == 0:
        diffevks = np.subtract(late1rdm_diff, early1rdm_diff)
        rdm1_diff.append(diffevks)
        rdm1_diff = rdm1_diff[0] #to keep shape of object consistent
    elif cond == 1:
        diffevks = np.subtract(late2rdm_diff, early2rdm_diff)
        rdm2_diff.append(diffevks)
        rdm2_diff = rdm2_diff[0]

In [10]:
# Generate a data frame to tabulate condition, cluster indices, cluster timepts, p values of PERTURBATION comparisons
# This information can then be included in ERP difference wave plots

conds = ['rdm1vrdm2']
conditionnames = ['rdm1vrdm2']
p = 0.05
perms = 1000

condition = []
clust_idx_start = []
clust_idx_end = []
time_start = []
time_end = []
p_values = []

for c in range(0, len(conds)):
    if c == 0:
        T_0, clust_idx, clust_pvals, H0 = get_clust_perm_test(rdm1_diff, rdm2_diff, p, perms)
#         print(clust_idx, clust_pvals)
        if len(clust_idx) == 0:
            condition.append(conditionnames[c])
            clust_idx_start.append(np.nan)
            clust_idx_end.append(np.nan)
            time_start.append(np.nan)
            time_end.append(np.nan)
            p_values.append(np.nan)
        else:
            for clust in range(0, len(clust_idx)):
                cluster = clust_idx[clust][0] #to get the slice sequence we need
        
                cluster_start = cluster.start
                clust_idx_start.append(cluster_start)
        
                cluster_end = cluster.stop
                clust_idx_end.append(cluster_end)
        
                time_idx_start = time[cluster_start]
                time_start.append(time_idx_start)
        
                time_idx_end = time[cluster_end - 1] #minus one because python indexing does not include ending value
                time_end.append(time_idx_end)
        
                clust_p = clust_pvals[clust]
                p_values.append(clust_p)
        
                condition.append(conditionnames[c])
        
perm_test = pd.DataFrame(
    {'condition': condition,
     'clust_idx_start': clust_idx_start,
     'clust_idx_end': clust_idx_end,
     'time_start': time_start,
     'time_end': time_end,
     'p_values': p_values})

perm_test_filename = os.path.join('F:/Documents/Science/MirRevAdaptEEG/data/', 'Permutation_test_RDMPerturbTypeComp_%s.csv' % (erps))
perm_test.to_csv(perm_test_filename)

stat_fun(H1): min=-0.225131 max=1.951073
Running initial clustering …
Found 0 clusters


C:\Users\Raphael\AppData\Local\Temp\ipykernel_1332\3044986056.py:10: RuntimeWarning: No clusters found, returning empty H0, clusters, and cluster_pv
  T_0, clust_idx, clust_pvals, H0 = mne.stats.permutation_cluster_1samp_test(data, threshold = thresh,
